In [1]:
import os
base = '/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT'
for folder in ['train', 'val', 'test']:
    path = os.path.join(base, folder)
    print(f"{folder}/")
    for sub in os.listdir(path):
        sub_path = os.path.join(path, sub)
        print(f"  {sub}/ → {len(os.listdir(sub_path))} files")

train/
  labels/ → 26869 files
  images/ → 26869 files
val/
  labels/ → 5758 files
  images/ → 5758 files
test/
  labels/ → 5758 files
  images/ → 5758 files


In [2]:
import os
from collections import Counter

base = '/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT'

countries = Counter()
for split in ['train', 'val', 'test']:
    for img in os.listdir(f'{base}/{split}/images'):
        countries[img.split('_')[0]] += 1

for c, n in countries.most_common():
    print(f"{c}: {n}")

Japan: 10506
Norway: 8161
India: 7706
United: 4805
China: 4378
Czech: 2829


In [3]:
import os, glob, shutil

base = '/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT'
out  = '/kaggle/working/RDD_SPLIT_4CLASS'

for split in ['train', 'val', 'test']:
    os.makedirs(f'{out}/{split}/labels', exist_ok=True)
    os.makedirs(f'{out}/{split}/images', exist_ok=True)

    for lbl_path in glob.glob(f'{base}/{split}/labels/*.txt'):
        new_lines = []
        for line in open(lbl_path):
            if not line.strip():
                continue
            parts = line.split()
            cid = parts[0]
            if cid == '3':          # drop Repair
                continue
            if cid == '4':          # remap Pothole -> 3
                parts[0] = '3'
            new_lines.append(' '.join(parts) + '\n')

        fname = os.path.basename(lbl_path)
        with open(f'{out}/{split}/labels/{fname}', 'w') as f:
            f.writelines(new_lines)

        # symlink corresponding image (fast, no copy)
        img_src = lbl_path.replace('labels', 'images').replace('.txt', '.jpg')
        img_dst = f'{out}/{split}/images/{os.path.basename(img_src)}'
        if os.path.exists(img_src) and not os.path.exists(img_dst):
            os.symlink(img_src, img_dst)

print("Done. New dataset at:", out)

Done. New dataset at: /kaggle/working/RDD_SPLIT_4CLASS


In [4]:
import glob
from collections import Counter

out = '/kaggle/working/RDD_SPLIT_4CLASS'
cls = Counter()
for split in ['train','val','test']:
    for f in glob.glob(f'{out}/{split}/labels/*.txt'):
        for line in open(f):
            if line.strip():
                cls[line.split()[0]] += 1

for cid in sorted(cls, key=int):
    print(f"Class {cid}: {cls[cid]}")

Class 0: 26016
Class 1: 11830
Class 2: 10617
Class 3: 6544


In [5]:
yaml_content = """
train: /kaggle/working/RDD_SPLIT_4CLASS/train/images
val: /kaggle/working/RDD_SPLIT_4CLASS/val/images
test: /kaggle/working/RDD_SPLIT_4CLASS/test/images

nc: 4
names: ['D00', 'D10', 'D20', 'D40']
"""

with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(yaml_content)

print(open('/kaggle/working/data.yaml').read())


train: /kaggle/working/RDD_SPLIT_4CLASS/train/images
val: /kaggle/working/RDD_SPLIT_4CLASS/val/images
test: /kaggle/working/RDD_SPLIT_4CLASS/test/images

nc: 4
names: ['D00', 'D10', 'D20', 'D40']



In [6]:
import os
print(os.path.exists('/kaggle/working/RDD_SPLIT_4CLASS'))
print(os.path.exists('/kaggle/working/data.yaml'))

True
True


In [7]:
# ============================================
# Clone FRDC repo (needs Internet ON in notebook settings)
# ============================================
!git clone https://github.com/FangjunWang/FRDC_RDD.git /kaggle/working/FRDC_RDD

Cloning into '/kaggle/working/FRDC_RDD'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 139 (delta 13), reused 8 (delta 8), pack-reused 122 (from 2)
Receiving objects: 100% (139/139), 3.74 MiB | 18.75 MiB/s, done.
Resolving deltas: 100% (52/52), done.


# ============================================
# Install MMDetection 3.3.0 stack (separate from Ultralytics)
# ============================================

In [8]:
# STEP 1: torch 2.2.0 + cu121 (confirmed available from your earlier error)
!pip install -q torch==2.2.0 torchvision==0.17.0 --index-url https://download.pytorch.org/whl/cu121

# STEP 2: mmengine + mmcv 2.2.0 (has real prebuilt wheel for torch2.2/cu121)
!pip install -q mmengine
!pip install -q mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.2/index.html

# STEP 3: mmdet 3.3.0
!pip install -q mmdet==3.3.0

# STEP 4: patch mmdet's version-check assertion (mmcv 2.2.0 works fine with mmdet 3.3.0)
!sed -i "s/mmcv_maximum_version = '2.2.0'/mmcv_maximum_version = '2.3.0'/" /usr/local/lib/python3.12/dist-packages/mmdet/__init__.py

# Verify
import torch, mmcv, mmdet
print("torch:", torch.__version__, torch.cuda.is_available())
print("mmcv:", mmcv.__version__)
print("mmdet:", mmdet.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 2.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 60.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 91.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 213.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 84.5 MB/s eta 0:00:0000:010:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 44.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 64.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 79.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 133.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 108.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 77.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-packag

torch: 2.2.0+cu121 True
mmcv: 2.2.0
mmdet: 3.3.0


In [9]:
!pip install -q "numpy<2" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
c

In [10]:
import torch, mmcv, mmdet, numpy
print("torch:", torch.__version__, torch.cuda.is_available())
print("mmcv:", mmcv.__version__)
print("mmdet:", mmdet.__version__)
print("numpy:", numpy.__version__)

torch: 2.2.0+cu121 True
mmcv: 2.2.0
mmdet: 3.3.0
numpy: 2.0.2


In [11]:
import os
os.makedirs('/kaggle/working/weights', exist_ok=True)

!wget "https://huggingface.co/WangFangjun/FRDC-RDD/resolve/main/RTMDet/best_coco_bbox_mAP_epoch_292.pth" -O /kaggle/working/weights/rtmdet.pth

!ls -la /kaggle/working/weights

--2026-09-02 20:57:37--  https://huggingface.co/WangFangjun/FRDC-RDD/resolve/main/RTMDet/best_coco_bbox_mAP_epoch_292.pth
Resolving huggingface.co (huggingface.co)... 108.138.94.52, 108.138.94.97, 108.138.94.45, ...
Connecting to huggingface.co (huggingface.co)|108.138.94.52|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66e44f1b11e4f40aa79e69a8/d5bed26c0df2f319199e41c1725d08de0b979a1ce37f1b241b0f1f9d7557bc06?user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27best_coco_bbox_mAP_epoch_292.pth%3B+filename%3D%22best_coco_bbox_mAP_epoch_292.pth%22%3B&X-Xet-Cas-Uid=public&Expires=1788386258&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZlNDRmMWIxMWU0ZjQwYWE3OWU2OWE4L2Q1YmVkMjZjMGRmMmYzMTkxOTllNDFjMTcyNWQwOGRlMGI5NzlhMWNlMzdmMWIyNDFiMGYxZjlkNzU1N2JjMDZcXD91c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2FzLVVpZD1wdWJsaWMiLCJ

In [12]:
# ============================================
# List available configs to find the EXACT matching config filename
# (don't guess — pick the file that corresponds to Swin-L / RTMDet-x)
# ============================================
!ls /kaggle/working/FRDC_RDD/RTMDet/configs/rtmdet/

cspnext_imagenet_pretrain
distillation
metafile.yml
README.md
rotated
rtmdet-ins_s_syncbn_fast_8xb32-300e_coco.py
rtmdet_l_syncbn_fast_8xb32-300e_coco.py
rtmdet_m_syncbn_fast_8xb32-300e_coco.py
rtmdet_s_syncbn_fast_8xb32-300e_coco.py
rtmdet_tiny_fast_1xb12-40e_cat.py
rtmdet_tiny_syncbn_fast_8xb32-300e_coco.py
rtmdet_x_syncbn_fast_8xb32-300e_coco.py


In [13]:
# Install mmyolo itself only — don't let it touch existing packages
!pip install -q mmyolo --no-deps

# Find where it installed
!pip show mmyolo | grep Location

# Patch mmyolo's overly strict mmcv version check (same trick as mmdet earlier)
!sed -i "s/mmcv_maximum_version = '2.1.0'/mmcv_maximum_version = '2.3.0'/" /usr/local/lib/python3.12/dist-packages/mmyolo/__init__.py

# Also relax its mmdet upper bound, just in case
!sed -i "s/mmdet_maximum_version = '3.1.0'/mmdet_maximum_version = '3.4.0'/" /usr/local/lib/python3.12/dist-packages/mmyolo/__init__.py

# Verify
import mmyolo
print("mmyolo:", mmyolo.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.7/453.7 kB 11.4 MB/s eta 0:00:0000:01
Location: /usr/local/lib/python3.12/dist-packages
mmyolo: 0.6.0


In [14]:
!sed -i '/from mmdet.apis import init_detector, inference_detector/a from mmyolo.utils import register_all_modules\nregister_all_modules()' /kaggle/working/FRDC_RDD/RTMDet/infer.py

In [15]:
# ============================================
# CELL 7 — Run RTMDet inference (generates pseudo labels)
# Replace <config_filename> with what you saw printed in Cell 5
# ============================================
# Create the output folder for RTMDet's predictions, if it doesn't already exist

import os
os.makedirs('/kaggle/working/pseudo_labels/rtmdet', exist_ok=True)

# Run RTMDet inference using its infer.py script (from the FRDC repo)
# --config: model architecture (RTMDet-x)
# --ckpt: trained weights (FRDC's RDDv2 RTMDet checkpoint)
# --img_dir: images to predict on (your train split)
# --out_dir: where predictions get saved (one .txt file per image)
# --device: run on GPU 0
!python /kaggle/working/FRDC_RDD/RTMDet/infer.py \
    --config /kaggle/working/FRDC_RDD/RTMDet/configs/rtmdet/rtmdet_x_syncbn_fast_8xb32-300e_coco.py \
    --ckpt /kaggle/working/weights/rtmdet.pth \
    --img_dir /kaggle/working/RDD_SPLIT_4CLASS/train/images \
    --out_dir /kaggle/working/pseudo_labels/rtmdet \
    --device cuda:0

Loads checkpoint by local backend from path: /kaggle/working/weights/rtmdet.pth
  0%|                                                 | 0/26869 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
100%|███████████████████████████████████| 26869/26869 [1:14:11<00:00,  6.04it/s]


In [16]:
import os

# Zip the RTMDet pseudo labels
!zip -r /kaggle/working/rtmdet_pseudo_labels.zip /kaggle/working/pseudo_labels/rtmdet

# Verify the count
print(len(os.listdir('/kaggle/working/pseudo_labels/rtmdet')))

  adding: kaggle/working/pseudo_labels/rtmdet/ (stored 0%)
  adding: kaggle/working/pseudo_labels/rtmdet/Japan_002617.txt (deflated 57%)
  adding: kaggle/working/pseudo_labels/rtmdet/Norway_005029.txt (deflated 57%)
  adding: kaggle/working/pseudo_labels/rtmdet/Czech_003185.txt (deflated 56%)
  adding: kaggle/working/pseudo_labels/rtmdet/Japan_003637.txt (deflated 57%)
  adding: kaggle/working/pseudo_labels/rtmdet/China_Drone_000921.txt (deflated 57%)
  adding: kaggle/working/pseudo_labels/rtmdet/Japan_005626.txt (deflated 56%)
  adding: kaggle/working/pseudo_labels/rtmdet/Japan_000729.txt (deflated 58%)
  adding: kaggle/working/pseudo_labels/rtmdet/India_001585.txt (deflated 56%)
  adding: kaggle/working/pseudo_labels/rtmdet/India_006997.txt (deflated 56%)
  adding: kaggle/working/pseudo_labels/rtmdet/China_Drone_000920.txt (deflated 56%)
  adding: kaggle/working/pseudo_labels/rtmdet/Norway_001632.txt (deflated 57%)
  adding: kaggle/working/pseudo_labels/rtmdet/India_005883.txt (defla